# Final submission (Kaggle T4): re-read two test images with the stored models

Optional step of `pipeline.ipynb` for the final submission (`MODE = "ms125"` and the other chain modes the repository holds), on a Kaggle **T4** runtime. No training. It loads the models stored in this repository (`weights/final/`: the five crop-1536 Mask2Former fold models and the ten U-Net fold checkpoints, reassembled from parts and hash-checked), runs them on two test images exactly as the stored maps were made (Mask2Former: 8 flips and rotations at 2048 px and 2 views at 2560 px per model; U-Nets: each fold model), and compares the new maps with the stored ones, pixel by pixel.

**Why a comparison and not a hash.** The stored maps were made on an RTX 3090 (PyTorch 2.5.1, native BF16); a T4 has no native BF16 and the pinned environment is PyTorch 2.14.1, so a few pixels differ by a level or more. The CSV of `pipeline.ipynb` is built from the stored maps, so it is exact; this step shows that the stored models give those maps, and how closely.

**All 180 images.** Set `CHAIN_MAPS_ALL = True` before running to re-read every test image (several T4-hours); `reproduce_submission.ipynb` can then build the CSV from the new maps with `--maps-from-models` (see the reproduction notes).

In [ ]:
from pathlib import Path
import os, subprocess, sys, json
REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# Clone this repository at its release tag (needs Internet), unless this notebook already runs inside a checkout.
SOURCE_TAG = "v1.1"
if not (REPO / "src/filseg/reproduction").is_dir():
    REPO = Path("/kaggle/working/filament-reproduction-source")
    if not REPO.exists():
        subprocess.run(["git", "clone", "--no-checkout", "https://github.com/Matapanino/filament-segmentation-2026-solution.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", SOURCE_TAG], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(REPO)], check=True)
sys.path.insert(0, str(REPO / "src"))  # the editable install is seen only by new processes; the next cell imports filseg here

In [ ]:
from filseg.reproduction.modes import DEFAULT_MODE, resolve
from filseg.reproduction.paths import find_test_images
MODE = globals().get("MODE") or DEFAULT_MODE  # pipeline.ipynb sets MODE before running this notebook
MODE, MODE_ROW = resolve(MODE)
CHAIN_MAPS_ALL = globals().get("CHAIN_MAPS_ALL", False)  # True: all 180 test images (several T4-hours)
CHECK_IMAGES = ["20140420195854Bh", "20111122063134Lh"]  # the two images pipeline.ipynb draws
IMAGE_ROOT = find_test_images()
CHAIN_MAPS = Path("/kaggle/working/chain_maps")
command = [sys.executable, "-m", "filseg.reproduction", "chain-maps", "--images", str(IMAGE_ROOT), "--recipe", MODE, "--output", str(CHAIN_MAPS)]
if not CHAIN_MAPS_ALL:
    command += ["--image-keys", *CHECK_IMAGES]
run = subprocess.run(command)
if run.returncode != 0 or not (CHAIN_MAPS / "chain_maps_compare.json").is_file():
    # An optional check: the CSV stage that follows reads the stored maps and does not depend on it.
    print("The from-models check did not finish (exit code", run.returncode, "); see", CHAIN_MAPS, "- continuing with the stored maps")
else:
    compare = json.loads((CHAIN_MAPS / "chain_maps_compare.json").read_text())
    print({"images_compared": CHECK_IMAGES if not CHAIN_MAPS_ALL else "all", "all_equal": compare["all_equal"], "first_differing": compare["first_differing"]})
    for name, row in compare["summary"].items():
        print(name, row)